# NMF on ModulAir 00687

In [1]:
from sklearn.decomposition import NMF
import numpy as np
import pandas as pd

## Cleaning Data

In [2]:
#importing data from Modulair MOD-00068
data = pd.read_csv('MOD-00687.csv').set_index('timestamp')
data.head()

,id,timestamp_local,sn,rh,temp,bin0,bin1,bin2,bin3,bin4,...,no2,o3,pm1_model_id,pm25_model_id,pm10_model_id,co_model_id,no_model_id,no2_model_id,o3_model_id,ws_scalar
timestamp,,,,,,,,,,,,,,,,,,,,,
2025-12-27T14:09:49Z,574530654,2025-12-27T09:09:49Z,MOD-00687,76.0,-1.7,3.527,0.691,0.185,0.056,0.043,...,33.079,14.604,14334.0,14335.0,14336.0,14474.0,14499.0,14549.0,14524.0,2.48
2025-12-27T14:08:49Z,574530653,2025-12-27T09:08:49Z,MOD-00687,76.4,-1.8,3.761,0.627,0.218,0.041,0.078,...,32.818,14.558,14334.0,14335.0,14336.0,14474.0,14499.0,14549.0,14524.0,3.41
2025-12-27T14:07:49Z,574530652,2025-12-27T09:07:49Z,MOD-00687,76.5,-1.8,3.794,0.652,0.228,0.058,0.072,...,32.577,14.544,14334.0,14335.0,14336.0,14474.0,14499.0,14549.0,14524.0,5.03
2025-12-27T14:06:49Z,574530651,2025-12-27T09:06:49Z,MOD-00687,76.4,-1.8,4.033,0.634,0.246,0.053,0.042,...,32.581,14.558,14334.0,14335.0,14336.0,14474.0,14499.0,14549.0,14524.0,2.93
2025-12-27T14:05:49Z,574528724,2025-12-27T09:05:49Z,MOD-00687,77.4,-1.8,4.031,0.639,0.195,0.070,0.060,...,33.013,13.612,14334.0,14335.0,14336.0,14474.0,14499.0,14549.0,14524.0,4.50


In [3]:
#only columns of interest
COLS_TO_INCLUDE = ['timestamp_local','co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
data = data[COLS_TO_INCLUDE]

data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
timestamp,,,,,,,,,,,
2025-12-27T14:09:49Z,2025-12-27T09:09:49Z,783.067,1.723,33.079,14.604,3.527,0.691,0.185,0.056,0.043,0.052
2025-12-27T14:08:49Z,2025-12-27T09:08:49Z,781.854,1.919,32.818,14.558,3.761,0.627,0.218,0.041,0.078,0.083
2025-12-27T14:07:49Z,2025-12-27T09:07:49Z,792.866,1.919,32.577,14.544,3.794,0.652,0.228,0.058,0.072,0.031
2025-12-27T14:06:49Z,2025-12-27T09:06:49Z,783.496,1.919,32.581,14.558,4.033,0.634,0.246,0.053,0.042,0.048
2025-12-27T14:05:49Z,2025-12-27T09:05:49Z,812.342,1.976,33.013,13.612,4.031,0.639,0.195,0.070,0.060,0.040


In [4]:
#removing the UTC time
data = data.reset_index(drop = True)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-27T09:09:49Z,783.067,1.723,33.079,14.604,3.527,0.691,0.185,0.056,0.043,0.052
1,2025-12-27T09:08:49Z,781.854,1.919,32.818,14.558,3.761,0.627,0.218,0.041,0.078,0.083
2,2025-12-27T09:07:49Z,792.866,1.919,32.577,14.544,3.794,0.652,0.228,0.058,0.072,0.031
3,2025-12-27T09:06:49Z,783.496,1.919,32.581,14.558,4.033,0.634,0.246,0.053,0.042,0.048
4,2025-12-27T09:05:49Z,812.342,1.976,33.013,13.612,4.031,0.639,0.195,0.070,0.060,0.040


In [5]:
#converting to datetime
data['timestamp_local'] = pd.to_datetime(data['timestamp_local'],
                                       format='%Y-%m-%dT%H:%M:%SZ',
                                       exact=False)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-27 09:09:49,783.067,1.723,33.079,14.604,3.527,0.691,0.185,0.056,0.043,0.052
1,2025-12-27 09:08:49,781.854,1.919,32.818,14.558,3.761,0.627,0.218,0.041,0.078,0.083
2,2025-12-27 09:07:49,792.866,1.919,32.577,14.544,3.794,0.652,0.228,0.058,0.072,0.031
3,2025-12-27 09:06:49,783.496,1.919,32.581,14.558,4.033,0.634,0.246,0.053,0.042,0.048
4,2025-12-27 09:05:49,812.342,1.976,33.013,13.612,4.031,0.639,0.195,0.070,0.060,0.040


In [6]:
#taking hourly average of df. round to floor of the hour
data = data.groupby(data['timestamp_local'].dt.floor('h')).agg(co = ('co','mean'),
                                                         no2 = ('no2','mean'),
                                                         o3 = ('o3','mean'),
                                                         no = ('no','mean'),
                                                         bin0 = ('bin0','mean'),
                                                         bin1 = ('bin1','mean'),
                                                         bin2 = ('bin2','mean'),
                                                         bin3 = ('bin3','mean'),
                                                         bin4 = ('bin4','mean'),
                                                         bin5 = ('bin5','mean')).reset_index()

data = data.round(decimals = 2)
data = data.dropna()
data

,timestamp_local,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-05-15 08:00:00,819.33,12.64,21.05,2.58,53.62,8.31,2.61,0.64,0.61,0.26
1,2025-05-15 09:00:00,794.15,12.65,22.90,2.36,25.49,4.35,1.68,0.33,0.23,0.10
2,2025-05-15 10:00:00,752.28,15.83,24.76,1.89,15.77,3.53,1.56,0.24,0.13,0.06
3,2025-05-15 11:00:00,766.66,13.81,25.86,1.62,14.61,3.64,1.40,0.22,0.14,0.07
4,2025-05-15 12:00:00,984.10,16.52,25.38,1.75,16.83,5.37,2.30,0.39,0.23,0.09
...,...,...,...,...,...,...,...,...,...,...,...
5161,2025-12-27 05:00:00,816.94,33.34,11.88,2.73,3.39,0.57,0.20,0.06,0.06,0.05
5162,2025-12-27 06:00:00,801.46,33.23,12.18,2.50,3.68,0.58,0.21,0.06,0.05,0.04
5163,2025-12-27 07:00:00,801.97,32.72,12.00,2.27,4.57,0.77,0.28,0.07,0.08,0.07
5164,2025-12-27 08:00:00,761.54,32.62,13.74,1.99,3.83,0.65,0.22,0.06,0.06,0.05


In [7]:
#setting local time as index
data = data.set_index('timestamp_local')
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-05-15 08:00:00,819.33,12.64,21.05,2.58,53.62,8.31,2.61,0.64,0.61,0.26
2025-05-15 09:00:00,794.15,12.65,22.90,2.36,25.49,4.35,1.68,0.33,0.23,0.10
2025-05-15 10:00:00,752.28,15.83,24.76,1.89,15.77,3.53,1.56,0.24,0.13,0.06
2025-05-15 11:00:00,766.66,13.81,25.86,1.62,14.61,3.64,1.40,0.22,0.14,0.07
2025-05-15 12:00:00,984.10,16.52,25.38,1.75,16.83,5.37,2.30,0.39,0.23,0.09


In [8]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()

In [9]:
#scaling
def maximum_absolute_scaling(df):
    # copy the dataframe
    df_scaled = df.copy()
    # apply maximum absolute scaling
    for column in df_scaled.columns:
        df_scaled[column] = df_scaled[column]  / df_scaled[column].abs().max()
    return df_scaled

# call the maximum_absolute_scaling function
data = maximum_absolute_scaling(data)

In [10]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-05-15 08:00:00,0.428372,0.253256,0.247356,0.072391,0.628679,0.276908,0.145973,0.123314,0.136161,0.148571
2025-05-15 09:00:00,0.415207,0.253456,0.269095,0.066218,0.298863,0.144952,0.093960,0.063584,0.051339,0.057143
2025-05-15 10:00:00,0.393316,0.317171,0.290952,0.053030,0.184899,0.117627,0.087248,0.046243,0.029018,0.034286
2025-05-15 11:00:00,0.400834,0.276698,0.303878,0.045455,0.171298,0.121293,0.078300,0.042389,0.031250,0.040000
2025-05-15 12:00:00,0.514519,0.330996,0.298237,0.049102,0.197327,0.178940,0.128635,0.075145,0.051339,0.051429


In [11]:
data.to_csv(r'MOD-00687_timeseries_hourly_scaled.csv')

## Implementing NMF

In [12]:
#setting up the NMF
nmf = NMF(n_components = 4, max_iter = 8000)

In [13]:
W = nmf.fit_transform(data)
H = nmf.components_
V = pd.DataFrame(np.dot(W,H), columns=data.columns)
V.index = data.index
V

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-05-15 08:00:00,0.442000,0.248375,0.255827,0.052655,0.566102,0.367424,0.168269,0.111532,0.103620,0.125602
2025-05-15 09:00:00,0.412459,0.254382,0.272058,0.056305,0.288389,0.165644,0.075860,0.051436,0.054449,0.082082
2025-05-15 10:00:00,0.385783,0.319723,0.289371,0.056159,0.203734,0.106276,0.048671,0.033359,0.037660,0.061867
2025-05-15 11:00:00,0.392587,0.279252,0.300780,0.056937,0.197562,0.098526,0.045122,0.031230,0.036783,0.063397
2025-05-15 12:00:00,0.500658,0.335557,0.290908,0.070352,0.253188,0.136088,0.062324,0.043191,0.051014,0.088131
...,...,...,...,...,...,...,...,...,...,...
2025-12-27 05:00:00,0.425858,0.668875,0.140034,0.067008,0.041611,0.016571,0.007589,0.006377,0.015133,0.041105
2025-12-27 06:00:00,0.416316,0.666770,0.143929,0.065751,0.043435,0.017927,0.008210,0.006689,0.014875,0.039481
2025-12-27 07:00:00,0.418641,0.655681,0.140784,0.065547,0.055166,0.026235,0.012015,0.009213,0.017187,0.042182


In [14]:
W_df = pd.DataFrame(W, columns =[f'Feature {i+1}' for i in range(0,4)]) #array-like of shape (n_samples, n_components)
W_df

,Feature 1,Feature 2,Feature 3,Feature 4
0,0.010584,0.052630,0.221070,0.120938
1,0.016548,0.053908,0.099664,0.116708
2,0.024234,0.054756,0.063943,0.094142
3,0.020721,0.058655,0.059281,0.103597
4,0.024131,0.055137,0.081881,0.146104
...,...,...,...,...
5028,0.057812,0.008551,0.009970,0.077944
5029,0.057656,0.009476,0.010786,0.073025
5030,0.056412,0.009263,0.015785,0.075907
5031,0.056614,0.013719,0.011831,0.062821


In [15]:
COLS_TO_INCLUDE.pop(0)
COLS_TO_INCLUDE

['co', 'no', 'no2', 'o3', 'bin0', 'bin1', 'bin2', 'bin3', 'bin4', 'bin5']

In [16]:
H_df = pd.DataFrame(H, index = [f'Feature {i+1}' for i in range(0,4)], columns = COLS_TO_INCLUDE) #array-like of shape (n_components, n_features)
H_df

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Feature 1,4.338745,11.275119,1.755485,0.758342,0.000000,0.000000,0.000000,0.000000,0.033114,0.132207
Feature 2,1.096153,0.000000,4.507815,0.220596,0.778997,0.000000,0.000000,0.000000,0.000000,0.000000
Feature 3,0.395706,0.498989,0.000000,0.000000,2.290250,1.662025,0.761155,0.494349,0.402528,0.351570
Feature 4,2.074689,0.154819,0.000000,0.273024,0.155431,0.000000,0.000000,0.018574,0.118102,0.384342


In [17]:
#converting the results to a dataframe
results = pd.DataFrame(W,index=data.index) #H is time series data of the factors, W is the comp (coeff matrix)
results.columns = ["Factor {}".format(i+1) for i in range(H.T.shape[1])]
results

,Factor 1,Factor 2,Factor 3,Factor 4
timestamp_local,,,,
2025-05-15 08:00:00,0.010584,0.052630,0.221070,0.120938
2025-05-15 09:00:00,0.016548,0.053908,0.099664,0.116708
2025-05-15 10:00:00,0.024234,0.054756,0.063943,0.094142
2025-05-15 11:00:00,0.020721,0.058655,0.059281,0.103597
2025-05-15 12:00:00,0.024131,0.055137,0.081881,0.146104
...,...,...,...,...
2025-12-27 05:00:00,0.057812,0.008551,0.009970,0.077944
2025-12-27 06:00:00,0.057656,0.009476,0.010786,0.073025
2025-12-27 07:00:00,0.056412,0.009263,0.015785,0.075907


In [18]:
COLS_TO_INCLUDE = ['co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
comp = pd.DataFrame(H, index = results.columns, columns = COLS_TO_INCLUDE)
comp

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Factor 1,4.338745,11.275119,1.755485,0.758342,0.000000,0.000000,0.000000,0.000000,0.033114,0.132207
Factor 2,1.096153,0.000000,4.507815,0.220596,0.778997,0.000000,0.000000,0.000000,0.000000,0.000000
Factor 3,0.395706,0.498989,0.000000,0.000000,2.290250,1.662025,0.761155,0.494349,0.402528,0.351570
Factor 4,2.074689,0.154819,0.000000,0.273024,0.155431,0.000000,0.000000,0.018574,0.118102,0.384342


In [19]:
res = []

for col in comp.columns:
    #for each factor, compute its contribution to the species in V
    by_factor = pd.Series(dtype=float)
    for i, factor in enumerate(results.columns):
        contrib = H[i, data.columns.get_loc(col)] * W[:, i].sum()
        by_factor.at[factor] = contrib

    #normalizing by the total amount of that species in the original data
    by_factor /= data[col].sum()

    #adding as a row to the resulting dataframe
    res.append(pd.DataFrame(by_factor).T.rename(index={0: col}))

res = pd.concat(res)
res.columns = results.columns
res['Residual'] = 1 - res.sum(axis=1)

res

,Factor 1,Factor 2,Factor 3,Factor 4,Residual
co,0.477346,0.147076,0.034650,0.341110,-0.000182
no,0.525642,0.186476,0.000000,0.282812,0.005070
no2,0.947273,0.000000,0.033367,0.019438,-0.000077
o3,0.241984,0.757804,0.000000,0.000000,0.000212
bin0,0.000000,0.323750,0.621186,0.079156,-0.024092
bin1,0.000000,0.000000,1.039005,0.000000,-0.039005
bin2,0.000000,0.000000,1.060219,0.000000,-0.060219
bin3,0.000000,0.000000,0.945756,0.066720,-0.012476
bin4,0.062246,0.000000,0.602226,0.331761,0.003766
bin5,0.133044,0.000000,0.281592,0.578007,0.007357


In [20]:
results.to_csv(r'MOD-00687_4_factor_results.csv')
comp.to_csv(r'MOD-00687_4_factor_comp.csv')
res.to_csv(r'MOD-00687_4_factor_resid.csv')